## Spam detection using Navie Bayes

In [1]:
import numpy as np

### Collecting the Dataset

In [2]:
spam = [
    "To use your credit, click the new WAP link in the next years txt message or click here", 
    "Thanks for your subscription to New Ringtone UK your new mobile will be charged £5/month Please confirm annoncement by replying", 
    "As a valued customer, I am pleased to advise you that following recent delivery waiting review of your Mob No. you are awarded with. Call us to review.", 
    "Please call our new customer service representative on", 
    "We are trying to contact you. Last weekends customer draw shows that you won a £1000 prize GUARANTEED. Calling years", 
]

In [3]:
# leave one sentence from spam for testing our model later (test set)
spam_test = ["Customer service annoncement. You have a New Years delivery waiting for you. click"]

In [4]:
non = [
    "I don't think he goes to usf, he lives around here though", 
    "New car and house for my parents. i have only new job in hand", 
    "Great escape. I fancy the bridge but needs her lager. See you tomorrow", 
    "Tired. I haven't slept well the past few nights.",
    "Too late. I said i have the website. I didn't i have or dont have the slippers", 
    "I might come by tonight then if my class lets out early", 
    "Jos ask if u wana meet up?", 
    "That would be great. We'll be at the Guild. We can try meeting with the customer on Bristol road or somewhere"
    ]

In [5]:
# another sentence from non for testing our model 
spam_test.append("That would be great. We'll be at the Guild. We can try meeting with the customer on Bristol road or somewhere")

# Basic Pre-Processing

as we have english words in text so we have to do some pre-processign\
i.e. we have to convert the all the words to lower case and to the stem word\
also have to remove stop words. this is basic NLP pre-processing

In [6]:
# !pip install nltk

In [7]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

# nltk.download('stopwords')
# nltk.download('punkt')
# nltk.download('punkt_tab')

In [8]:
test_sentence = non[4]
# test_sentence = non[5]
test_sentence = spam[1]
stop_words_english =  set(stopwords.words('english'))

# tokkenize the sentence
tokkens = word_tokenize(test_sentence.lower())
# removing stop words
tokkens = [word for word in tokkens if word not in stop_words_english]
# stem the words
p = PorterStemmer()
tokkens = [p.stem(word) for word in tokkens]

print(test_sentence)
print(tokkens)

Thanks for your subscription to New Ringtone UK your new mobile will be charged £5/month Please confirm annoncement by replying
['thank', 'subscript', 'new', 'rington', 'uk', 'new', 'mobil', 'charg', '£5/month', 'pleas', 'confirm', 'annonc', 'repli']


## Create dictionary of wrods

In [9]:
stop_words_english = set(stopwords.words('english'))
def tokkenize_sentence(sentence):
    p = PorterStemmer()
    tokkens = word_tokenize(sentence.lower())
    tokkens = [p.stem(word) for word in tokkens if word not in stop_words_english]
    return list(tokkens)

In [10]:
dictionary = set() # will have unique words only
spams_tokenized = []
nons_tokenized = []

for sentence in spam:
    sentence_tokkens = tokkenize_sentence(sentence)
    spams_tokenized.append(sentence_tokkens)
    dictionary = dictionary.union(sentence_tokkens)
    
for sentence in non:
    sentence_tokkens = tokkenize_sentence(sentence)
    nons_tokenized.append(sentence_tokkens)
    dictionary = dictionary.union(sentence_tokkens)

print(f"Dictionary: {dictionary}\n")
print(f"Spam tokkenized: {spams_tokenized}\n")
print(f"Not Spam tokkenized: {nons_tokenized}\n")



Dictionary: {'wana', 'last', 'let', 'car', 'earli', 'deliveri', 'show', 'think', 'new', 'year', "'ll", 'said', 'wait', 'somewher', 'rington', 'bristol', 'servic', 'recent', ',', 'ask', 'messag', 'would', 'click', 'link', 'custom', 'escap', 'review', 'repli', 'repres', 'parent', 'award', 'night', 'u', 'wap', 'txt', 'well', 'come', 'pleas', 'slipper', 'guild', 'confirm', 'weekend', 'contact', 'draw', '?', 'prize', 'mobil', 'around', 'thank', 'hous', 'credit', 'slept', 'need', 'use', 'goe', 'class', 'see', 'tire', 'live', 'us', 'charg', 'hand', 'though', 'guarante', 'dont', 'lager', 'great', 'subscript', 'advis', 'bridg', 'websit', 'valu', 'tomorrow', '£1000', 'late', 'mob', 'might', 'tonight', 'tri', 'past', '£5/month', 'jo', 'road', 'meet', 'uk', 'next', "n't", 'call', 'follow', '.', 'annonc', 'fanci', 'usf', 'job'}

Spam tokkenized: [['use', 'credit', ',', 'click', 'new', 'wap', 'link', 'next', 'year', 'txt', 'messag', 'click'], ['thank', 'subscript', 'new', 'rington', 'uk', 'new', 'mo

## Basic Stats

In [11]:
# these things don't depend upon individual word so we can calculate them seperately

total_word_count = len(dictionary)
total_spam_messages = len(spams_tokenized)
total_non_messages = len(nons_tokenized)
total_all_messages = total_non_messages + total_spam_messages

print(f"Total Number of words: {total_word_count}") 
print(f"Total spam messages: {total_spam_messages}")

Total Number of words: 94
Total spam messages: 5


In [12]:
# similarly probability of spam P(spam) does not depend upon the individual word

p_spam = total_spam_messages / total_all_messages

print(f"P(spam) = {p_spam}")

P(spam) = 0.38461538461538464


In [15]:
# helper funciton to count occurances

def count_word_in_messages(word, messages):
    count = 0
    for msg in messages:
        if word in msg: # count one time in a message only
            count += 1

    return count

### The Actual Probability Computation

In [24]:

for test_sentence in spam_test:
    ## tokkenize the test sentence
    test_tokkens = tokkenize_sentence(test_sentence)

    """ 
        we need to find P(spam|test_tokkens) 
        whcih is equal to 
            [ P(test_tokkens|spam) * P(spam) ] / P(test_tokkens)
    """

    ## lets calculate P(test_tokkens|spam). => p_tt_spam as words are independent we can product P of each tokken
    p_tt_spam = 1
    for tokken in test_tokkens:
        count_spam = count_word_in_messages(tokken, spams_tokenized)
        p_tt_spam *= (count_spam + 1) / (total_spam_messages + total_word_count) # use laplase smoothing


    ## lets calculate P(test_tokkens) => p_tt
    p_tt = 1
    for tokken in test_tokkens:
        count_tt = count_word_in_messages(tokken, spams_tokenized)
        count_tt += count_word_in_messages(tokken, nons_tokenized)

        p_tt *= (count_tt + 1) / (total_all_messages + total_word_count) # use laplase smoothing

    ## now we can calculate final probabilty
    p_spam_tt = (p_tt_spam * p_spam) / p_tt

    print(f"p_tt_spam: {p_tt_spam}")
    print(f"p_tt: {p_tt}")
    print(f"p_spam: {p_spam}")
    
    print(f"Teset Sentence:\n {test_sentence}")
    print(f"Proabibility that this is spam: {p_spam_tt}")
    

p_tt_spam: 1.5285574959969688e-16
p_tt: 7.80824512718926e-16
p_spam: 0.38461538461538464
Teset Sentence:
 Customer service annoncement. You have a New Years delivery waiting for you. click
Proabibility that this is spam: 0.0752930677320107
p_tt_spam: 8.12288231641419e-23
p_tt: 2.455208529817568e-19
p_spam: 0.38461538461538464
Teset Sentence:
 That would be great. We'll be at the Guild. We can try meeting with the customer on Bristol road or somewhere
Proabibility that this is spam: 0.000127247256938509


In [25]:
print(total_spam_messages)

5


In [28]:


final_prob = 1   # can't start from 0 


for test_sentence in spam_test: 
    test_sentence = tokkenize_sentence(test_sentence)
    print(test_sentence)
    
    # let's run this for each word separately 
    for word in test_sentence: 
        print("----------------")
        print("Runnig for word:", word)
        
        # Find P( w | spam)
        spam_count = count_word_in_messages(word, spams_tokenized)
        p_w_spam = spam_count / total_spam_messages 
        print("P( w | spam)  = ", p_w_spam)
        
        # Find P( w )
        w_count = count_word_in_messages(word, spams_tokenized)
        w_count += count_word_in_messages(word, nons_tokenized)
        p_w = w_count / total_all_messages
        print("P( w )        = ", p_w)
        
        
        # Find P( spam | w )
        p_spam_w = (p_w_spam * p_spam) / p_w
        print("P( spam )     = ", p_spam)
        print("P( spam | w ) = ", p_spam_w)
        print("")
        final_prob *= p_spam_w
        
        
    print("P( spam | all_words ) = ", final_prob)



['custom', 'servic', 'annonc', '.', 'new', 'year', 'deliveri', 'wait', '.', 'click']
----------------
Runnig for word: custom
P( w | spam)  =  0.6
P( w )        =  0.3076923076923077
P( spam )     =  0.38461538461538464
P( spam | w ) =  0.75

----------------
Runnig for word: servic
P( w | spam)  =  0.2
P( w )        =  0.07692307692307693
P( spam )     =  0.38461538461538464
P( spam | w ) =  1.0

----------------
Runnig for word: annonc
P( w | spam)  =  0.2
P( w )        =  0.07692307692307693
P( spam )     =  0.38461538461538464
P( spam | w ) =  1.0

----------------
Runnig for word: .
P( w | spam)  =  0.4
P( w )        =  0.5384615384615384
P( spam )     =  0.38461538461538464
P( spam | w ) =  0.28571428571428575

----------------
Runnig for word: new
P( w | spam)  =  0.6
P( w )        =  0.3076923076923077
P( spam )     =  0.38461538461538464
P( spam | w ) =  0.75

----------------
Runnig for word: year
P( w | spam)  =  0.4
P( w )        =  0.15384615384615385
P( spam )     =  0.38